# Whiskey Oracle: Multi-Task Prediction from Tasting Notes

Fine-tunes Llama 3.1 8B using QLoRA to predict whiskey attributes (ABV, Rating, Price, Type) from tasting notes.

## Setup

In [1]:
# Install required packages
!pip install -q --upgrade torch==2.5.1+cu124 torchvision==0.20.1+cu124 torchaudio==2.5.1+cu124 --index-url https://download.pytorch.org/whl/cu124
!pip install -q --upgrade transformers==4.48.3 accelerate==1.3.0 datasets==3.2.0 peft==0.14.0 trl==0.14.0 bitsandbytes==0.46.0
!pip install -q --upgrade pandas scikit-learn matplotlib seaborn wandb requests

zsh:1: command not found: pip
zsh:1: command not found: pip
zsh:1: command not found: pip


In [ ]:
import os
import re
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from datetime import datetime

import torch
import torch.nn.functional as F
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from datasets import Dataset, DatasetDict, load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

try:
    from google.colab import userdata
    from huggingface_hub import login
    import wandb
    IN_COLAB = True
except:
    IN_COLAB = False

set_seed(42)
%matplotlib inline

## Configuration

In [ ]:
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
QUANT_4_BIT = True

PROJECT_NAME = "whiskey-oracle"
HF_USER = "dc_dalin"
RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

TRAIN_SIZE = 500
VAL_SPLIT = 0.15
TEST_SIZE = 100

USE_WANDB = True

print(f"Project: {PROJECT_NAME}")
print(f"Run: {RUN_NAME}")
print(f"Hub Model: {HUB_MODEL_NAME}")

## Authentication

In [ ]:
if IN_COLAB:
    hf_token = userdata.get('HF_TOKEN')
    login(hf_token, add_to_git_credential=True)
else:
    from huggingface_hub import login
    login()

print("✅ HuggingFace authenticated")

In [ ]:
if USE_WANDB:
    if IN_COLAB:
        wandb_api_key = userdata.get('WANDB_API_KEY')
        os.environ["WANDB_API_KEY"] = wandb_api_key
    
    import wandb
    wandb.login()
    
    os.environ["WANDB_PROJECT"] = PROJECT_NAME
    os.environ["WANDB_LOG_MODEL"] = "checkpoint"
    os.environ["WANDB_WATCH"] = "gradients"
    
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)
    print("✅ W&B initialized")
else:
    print("⏭️  W&B disabled")

## Data Loading

In [ ]:
def load_whiskey_data():
    try:
        url = "https://raw.githubusercontent.com/makispl/Machine-Learning-Whiskey-Dataset/main/whiskey_data.csv"
        df = pd.read_csv(url)
        required_cols = {'age', 'abv', 'type'}
        if not required_cols.issubset(df.columns):
            print(f"⚠️  Dataset missing required columns, falling back to synthetic dataset")
        else:
            print(f"✅ Loaded {len(df)} whiskeys from GitHub")
            return df
    except Exception as e:
        print(f"⚠️  Creating synthetic dataset...")
    
    whiskey_types = ['Single Malt Scotch', 'Blended Scotch', 'Bourbon', 'Irish Whiskey', 'Rye', 'Japanese Whisky']
    
    descriptors_by_type = {
        'Single Malt Scotch': [
            "Rich and complex with notes of peat smoke, dried fruit, and sea salt. Long, warming finish with hints of oak and spice.",
            "Smooth and elegant with flavors of honey, vanilla, and light citrus. Delicate finish with subtle oak influence.",
            "Bold and peaty with intense smoke, iodine, and medicinal notes. Powerful finish with maritime character.",
            "Fruity and sherried with notes of dark chocolate, dried figs, and Christmas cake. Rich, lingering finish."
        ],
        'Bourbon': [
            "Sweet and full-bodied with caramel, vanilla, and toasted oak. Smooth finish with hints of butterscotch and spice.",
            "Rich and robust with notes of corn sweetness, maple, and cinnamon. Warm, spicy finish with charred oak.",
            "Mellow and approachable with honey, brown sugar, and light fruit. Easy finish with gentle warmth."
        ],
        'Irish Whiskey': [
            "Light and smooth with notes of vanilla, honey, and green apple. Clean, crisp finish.",
            "Creamy and mild with flavors of malt, light spice, and subtle fruit. Gentle, smooth finish."
        ],
        'Japanese Whisky': [
            "Delicate and refined with notes of mizunara oak, sandalwood, and light fruit. Elegant, balanced finish.",
            "Harmonious blend of malt and grain with subtle smoke, vanilla, and floral notes. Clean, sophisticated finish."
        ],
        'Rye': [
            "Spicy and bold with rye grain, black pepper, and mint. Assertive finish with lingering spice.",
            "Robust and flavorful with notes of cinnamon, clove, and dried fruit. Warm, peppery finish."
        ]
    }
    
    data = []
    np.random.seed(42)
    
    for i in range(1000):
        whiskey_type = np.random.choice(whiskey_types)
        
        if whiskey_type == 'Single Malt Scotch':
            age = np.random.choice([10, 12, 15, 18, 21, 25], p=[0.3, 0.3, 0.2, 0.1, 0.07, 0.03])
            abv = np.random.uniform(40, 60)
            base_price = 50 + (age - 10) * 8 + np.random.normal(0, 15)
            rating = 70 + age * 0.8 + np.random.normal(0, 5)
        elif whiskey_type == 'Bourbon':
            age = np.random.choice([4, 6, 8, 10, 12], p=[0.3, 0.3, 0.2, 0.15, 0.05])
            abv = np.random.uniform(40, 50)
            base_price = 30 + age * 5 + np.random.normal(0, 10)
            rating = 75 + age * 0.5 + np.random.normal(0, 5)
        elif whiskey_type == 'Japanese Whisky':
            age = np.random.choice([12, 15, 18, 21], p=[0.4, 0.3, 0.2, 0.1])
            abv = np.random.uniform(43, 55)
            base_price = 80 + age * 10 + np.random.normal(0, 20)
            rating = 80 + age * 0.7 + np.random.normal(0, 4)
        else:
            age = np.random.choice([4, 6, 8, 10, 12], p=[0.3, 0.3, 0.2, 0.15, 0.05])
            abv = np.random.uniform(40, 48)
            base_price = 35 + age * 4 + np.random.normal(0, 10)
            rating = 72 + age * 0.6 + np.random.normal(0, 5)
        
        if whiskey_type in descriptors_by_type:
            description = np.random.choice(descriptors_by_type[whiskey_type])
        else:
            description = np.random.choice(descriptors_by_type['Single Malt Scotch'])
        
        full_description = f"{age} year old {whiskey_type}. {description}"
        
        data.append({
            'name': f"{whiskey_type} {i+1}",
            'type': whiskey_type,
            'age': int(age),
            'abv': round(abv, 1),
            'rating': max(60, min(100, round(rating, 1))),
            'price': max(20, round(base_price, 2)),
            'description': full_description
        })
    
    df = pd.DataFrame(data)
    print(f"✅ Created synthetic dataset with {len(df)} whiskeys")
    return df

df_whiskey = load_whiskey_data()
df_whiskey.head()

In [ ]:
print("Dataset shape:", df_whiskey.shape)
print("\nBasic statistics:")
df_whiskey[['age', 'abv', 'rating', 'price']].describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

df_whiskey['type'].value_counts().plot(kind='bar', ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Whiskey Types')
axes[0, 0].tick_params(axis='x', rotation=45)

axes[0, 1].hist(df_whiskey['abv'], bins=30, color='coral', edgecolor='black')
axes[0, 1].set_title('ABV Distribution')

axes[0, 2].hist(df_whiskey['rating'], bins=30, color='lightgreen', edgecolor='black')
axes[0, 2].set_title('Rating Distribution')

axes[1, 0].hist(df_whiskey['price'], bins=30, color='gold', edgecolor='black')
axes[1, 0].set_title('Price Distribution')

df_whiskey['age'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 1], color='mediumpurple')
axes[1, 1].set_title('Age Distribution')

axes[1, 2].scatter(df_whiskey['rating'], df_whiskey['price'], alpha=0.5, color='steelblue')
axes[1, 2].set_title('Price vs Rating')

plt.tight_layout()
plt.show()

## Preprocessing

In [ ]:
def create_multi_task_prompt(row, include_output=True):
    prompt = f"""Analyze this whiskey and predict its characteristics:

Description: {row['description']}

Predictions:"""
    
    if include_output:
        output = f""" Type: {row['type']} | ABV: {row['abv']}% | Rating: {row['rating']}/100 | Price: ${row['price']}"""
        return prompt + output
    else:
        return prompt

sample = df_whiskey.iloc[0]
print(create_multi_task_prompt(sample))

In [ ]:
df_whiskey['text'] = df_whiskey.apply(create_multi_task_prompt, axis=1)

train_val_df = df_whiskey.sample(n=min(TRAIN_SIZE, len(df_whiskey)), random_state=42)
test_df = df_whiskey.drop(train_val_df.index).sample(n=min(TEST_SIZE, len(df_whiskey) - len(train_val_df)), random_state=42)

train_df, val_df = train_test_split(train_val_df, test_size=VAL_SPLIT, random_state=42)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

## Model Setup

In [ ]:
if not torch.cuda.is_available():
    print("❌ WARNING: No GPU detected! Enable GPU in Runtime → Change runtime type")
    QUANT_4_BIT = False
    quant_config = None
else:
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    
    if QUANT_4_BIT:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4"
        )
    else:
        quant_config = BitsAndBytesConfig(load_in_8bit=True)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✅ Tokenizer loaded")

In [ ]:
if quant_config is None:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        device_map="cpu",
        trust_remote_code=True,
        torch_dtype=torch.float32
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=True
    )

base_model.generation_config.pad_token_id = tokenizer.pad_token_id

if quant_config is not None:
    base_model = prepare_model_for_kbit_training(base_model)

print(f"✅ Model loaded")

In [ ]:
try:
    LoraConfig
except NameError:
    from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)

base_model = get_peft_model(base_model, lora_config)
base_model.print_trainable_parameters()

## Training

In [ ]:
response_template = "Predictions:"
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

In [ ]:
EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 512

training_args = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    run_name=RUN_NAME,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_32bit",
    weight_decay=0.001,
    max_grad_norm=0.3,
    
    fp16=False,
    bf16=True,
    
    eval_strategy="steps",
    eval_steps=50,
    per_device_eval_batch_size=2,
    
    logging_steps=20,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    report_to="wandb" if USE_WANDB else "none",
    push_to_hub=False,
)

In [ ]:
trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
# Start training
print("🚀 Starting fine-tuning...\n")
trainer.train()
print("\n✅ Training complete!")

In [ ]:
trainer.train()

In [ ]:
trainer.model.save_pretrained(f"./{PROJECT_RUN_NAME}-final")
tokenizer.save_pretrained(f"./{PROJECT_RUN_NAME}-final")

if USE_WANDB:
    wandb.finish()

## Inference

In [ ]:
fine_tuned_model = trainer.model
fine_tuned_model.eval()

In [ ]:
def parse_prediction(text):
    result = {
        'type': None,
        'abv': None,
        'rating': None,
        'price': None
    }
    
    type_match = re.search(r'Type:\s*([^|]+)', text)
    if type_match:
        result['type'] = type_match.group(1).strip()
    
    abv_match = re.search(r'ABV:\s*([\d.]+)', text)
    if abv_match:
        try:
            result['abv'] = float(abv_match.group(1))
        except:
            pass
    
    rating_match = re.search(r'Rating:\s*([\d.]+)', text)
    if rating_match:
        try:
            result['rating'] = float(rating_match.group(1))
        except:
            pass
    
    price_match = re.search(r'Price:\s*\$?([\d.]+)', text)
    if price_match:
        try:
            result['price'] = float(price_match.group(1))
        except:
            pass
    
    return result

In [ ]:
def predict_greedy(description, model, tokenizer, max_new_tokens=50):
    set_seed(42)
    prompt = create_multi_task_prompt({'description': description}, include_output=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return parse_prediction(response)

def predict_topk(description, model, tokenizer, max_new_tokens=50, top_k=10, temperature=0.7):
    set_seed(42)
    prompt = create_multi_task_prompt({'description': description}, include_output=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=top_k,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return parse_prediction(response)

def predict_beam(description, model, tokenizer, max_new_tokens=50, num_beams=4):
    set_seed(42)
    prompt = create_multi_task_prompt({'description': description}, include_output=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return parse_prediction(response)

test_sample = test_df.iloc[0]

print(f"Description: {test_sample['description']}\n")
print(f"Actual: Type={test_sample['type']} | ABV={test_sample['abv']}% | Rating={test_sample['rating']} | Price=${test_sample['price']}\n")

pred_greedy = predict_greedy(test_sample['description'], fine_tuned_model, tokenizer)
print(f"Greedy: Type={pred_greedy['type']} | ABV={pred_greedy['abv']}% | Rating={pred_greedy['rating']} | Price=${pred_greedy['price']}\n")

pred_topk = predict_topk(test_sample['description'], fine_tuned_model, tokenizer)
print(f"Top-k: Type={pred_topk['type']} | ABV={pred_topk['abv']}% | Rating={pred_topk['rating']} | Price=${pred_topk['price']}\n")

pred_beam = predict_beam(test_sample['description'], fine_tuned_model, tokenizer)
print(f"Beam: Type={pred_beam['type']} | ABV={pred_beam['abv']}% | Rating={pred_beam['rating']} | Price=${pred_beam['price']}")

## Evaluation

In [ ]:
def evaluate_strategy(predict_fn, test_data, strategy_name):
    predictions = []
    
    for idx, row in tqdm(test_data.iterrows(), total=len(test_data), desc=strategy_name):
        pred = predict_fn(row['description'], fine_tuned_model, tokenizer)
        predictions.append({
            'pred_type': pred['type'],
            'pred_abv': pred['abv'],
            'pred_rating': pred['rating'],
            'pred_price': pred['price'],
            'true_type': row['type'],
            'true_abv': row['abv'],
            'true_rating': row['rating'],
            'true_price': row['price']
        })
    
    results_df = pd.DataFrame(predictions)
    metrics = {}
    
    valid_abv = results_df.dropna(subset=['pred_abv', 'true_abv'])
    if len(valid_abv) > 0:
        metrics['abv_mae'] = mean_absolute_error(valid_abv['true_abv'], valid_abv['pred_abv'])
        metrics['abv_r2'] = r2_score(valid_abv['true_abv'], valid_abv['pred_abv'])
    
    valid_rating = results_df.dropna(subset=['pred_rating', 'true_rating'])
    if len(valid_rating) > 0:
        metrics['rating_mae'] = mean_absolute_error(valid_rating['true_rating'], valid_rating['pred_rating'])
        metrics['rating_r2'] = r2_score(valid_rating['true_rating'], valid_rating['pred_rating'])
    
    valid_price = results_df.dropna(subset=['pred_price', 'true_price'])
    if len(valid_price) > 0:
        metrics['price_mae'] = mean_absolute_error(valid_price['true_price'], valid_price['pred_price'])
        metrics['price_r2'] = r2_score(valid_price['true_price'], valid_price['pred_price'])
    
    valid_type = results_df.dropna(subset=['pred_type', 'true_type'])
    if len(valid_type) > 0:
        metrics['type_accuracy'] = (valid_type['pred_type'] == valid_type['true_type']).mean() * 100
    
    return metrics, results_df

In [ ]:
strategies = [
    ('Greedy', predict_greedy),
    ('Top-k', predict_topk),
    ('Beam', predict_beam)
]

all_metrics = {}
all_results = {}

for name, predict_fn in strategies:
    metrics, results = evaluate_strategy(predict_fn, test_df, name)
    all_metrics[name] = metrics
    all_results[name] = results

In [ ]:
metrics_df = pd.DataFrame(all_metrics).T
print(metrics_df)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

strategies_list = list(all_metrics.keys())
colors = ['#3498db', '#e74c3c', '#2ecc71']

abv_mae = [all_metrics[s].get('abv_mae', 0) for s in strategies_list]
axes[0, 0].bar(strategies_list, abv_mae, color=colors)
axes[0, 0].set_title('ABV MAE')

rating_mae = [all_metrics[s].get('rating_mae', 0) for s in strategies_list]
axes[0, 1].bar(strategies_list, rating_mae, color=colors)
axes[0, 1].set_title('Rating MAE')

price_mae = [all_metrics[s].get('price_mae', 0) for s in strategies_list]
axes[1, 0].bar(strategies_list, price_mae, color=colors)
axes[1, 0].set_title('Price MAE')

type_acc = [all_metrics[s].get('type_accuracy', 0) for s in strategies_list]
axes[1, 1].bar(strategies_list, type_acc, color=colors)
axes[1, 1].set_title('Type Accuracy (%)')

plt.tight_layout()
plt.show()

## Error Analysis

In [ ]:
best_strategy = min(all_metrics.keys(), key=lambda x: all_metrics[x].get('price_mae', float('inf')))
results = all_results[best_strategy].copy()

results['price_error'] = abs(results['pred_price'] - results['true_price'])
results['abv_error'] = abs(results['pred_abv'] - results['true_abv'])
results['rating_error'] = abs(results['pred_rating'] - results['true_rating'])

results = results.merge(test_df[['type', 'age', 'description']].reset_index(drop=True), left_index=True, right_index=True, suffixes=('', '_test'))

In [ ]:
error_by_type = results.groupby('type')[['price_error', 'abv_error', 'rating_error']].mean()
print(error_by_type)

error_by_type.plot(kind='bar', figsize=(12, 6), color=['#e74c3c', '#3498db', '#2ecc71'])
plt.title('Average Errors by Whiskey Type')
plt.xticks(rotation=45, ha='right')
plt.legend(['Price', 'ABV', 'Rating'])
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

valid_price = results.dropna(subset=['pred_price', 'true_price'])
axes[0].scatter(valid_price['true_price'], valid_price['pred_price'], alpha=0.6, color='#e74c3c')
max_price = max(valid_price['true_price'].max(), valid_price['pred_price'].max())
axes[0].plot([0, max_price], [0, max_price], 'b--', alpha=0.5)
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].set_title('Price')

valid_abv = results.dropna(subset=['pred_abv', 'true_abv'])
axes[1].scatter(valid_abv['true_abv'], valid_abv['pred_abv'], alpha=0.6, color='#3498db')
max_abv = max(valid_abv['true_abv'].max(), valid_abv['pred_abv'].max())
axes[1].plot([0, max_abv], [0, max_abv], 'b--', alpha=0.5)
axes[1].set_xlabel('Actual ABV')
axes[1].set_ylabel('Predicted ABV')
axes[1].set_title('ABV')

valid_rating = results.dropna(subset=['pred_rating', 'true_rating'])
axes[2].scatter(valid_rating['true_rating'], valid_rating['pred_rating'], alpha=0.6, color='#2ecc71')
axes[2].plot([60, 100], [60, 100], 'b--', alpha=0.5)
axes[2].set_xlabel('Actual Rating')
axes[2].set_ylabel('Predicted Rating')
axes[2].set_title('Rating')

plt.tight_layout()
plt.show()

print("Best Predictions:")
best_preds = results.nsmallest(3, 'price_error')
for idx, row in best_preds.iterrows():
    print(f"Actual: ${row['true_price']} | Predicted: ${row['pred_price']} | Error: ${row['price_error']:.2f}")

print("\nWorst Predictions:")
worst_preds = results.nlargest(3, 'price_error')
for idx, row in worst_preds.iterrows():
    print(f"Actual: ${row['true_price']} | Predicted: ${row['pred_price']} | Error: ${row['price_error']:.2f}")